## 1. Imports & Setup


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Display settings
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')


## 2. Load Dataset


In [9]:
DATA_PATH = '../global_bleaching_environmental.csv'

data = pd.read_csv(DATA_PATH, na_values=['nd', 'NA', ''])
print(f"Raw shape: {data.shape}")
print(f"\nFirst few rows:")

Raw shape: (41361, 62)

First few rows:


In [10]:
data.shape

(41361, 62)

In [11]:
drop_columns = ['Site_ID', 'Sample_ID', 'Data_Source','Ocean_Name', 'Reef_ID', 'Realm_Name',
       'Ecoregion_Name', 'Country_Name', 'State_Island_Province_Name',
       'City_Town_Name', 'Site_Name', 'Date_Day',
       'Substrate_Name', 'Site_Comments', 'Sample_Comments',
       'Bleaching_Comments', 'Bleaching_Level', 'Percent_Cover', 'SSTA_Mean', 'Exposure',
       ] # Exposure - interesting

data.drop(columns= drop_columns, inplace= True)


In [12]:
data = data[data['SSTA'] > 0]

data = data[data['Date_Year'] >= 1997]

PAPER_FEATURES = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']
TARGET = 'Percent_Bleaching'
data = data.dropna(subset=['Percent_Bleaching'])


In [13]:
data.shape

(21928, 42)

In [14]:
print(data[TARGET].isna().any().any())
print(data[PAPER_FEATURES].isna().any().any())

False
True


## 6. Train/Test Split


In [15]:
from sklearn.impute import SimpleImputer

# Train/test split ก่อน
X = data[PAPER_FEATURES]
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Impute (fit เฉพาะ train, transform ทั้งคู่)
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test  = imputer.transform(X_test)

## 7. LazyPredict Model Comparison (Optional)


In [51]:
try:
    from lazypredict.Supervised import LazyRegressor

    lazy_regressor = LazyRegressor(
        verbose=0,
        ignore_warnings=True,
        custom_metric=None
    )

    lazy_models, lazy_predictions = lazy_regressor.fit(
        X_train, X_test, y_train, y_test
    )

    print("LazyPredict model comparison (Top 15):")
    display(lazy_models.head(15))

except Exception as exc:
    lazy_models = None
    print("LazyPredict skipped.")
    print("ถ้าต้องการรัน cell นี้ ให้ติดตั้งก่อนด้วย: %pip install lazypredict")
    print(f"Reason: {type(exc).__name__}: {exc}")


LazyPredict model comparison (Top 15):


,Adjusted R-Squared,R-Squared,RMSE,Time Taken
Model,,,,
ExtraTreesRegressor,0.678075,0.678442,11.965608,1.463322
RandomForestRegressor,0.656095,0.656487,12.367350,4.067624
BaggingRegressor,0.624830,0.625258,12.917291,0.415238
XGBRegressor,0.578211,0.578692,13.696352,0.278676
DecisionTreeRegressor,0.490692,0.491272,15.050392,0.068410
HistGradientBoostingRegressor,0.481795,0.482386,15.181274,0.157610
ExtraTreeRegressor,0.460402,0.461017,15.491473,0.047251
GradientBoostingRegressor,0.353254,0.353991,16.959941,1.268361
KNeighborsRegressor,0.296851,0.297653,17.684023,0.081858


# Optuna

In [16]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, VotingRegressor

In [53]:
def et_objective(trial):

    model = ExtraTreesRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 300),
        max_depth=trial.suggest_int("max_depth", 5, 40),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1,
        random_state=42
    )

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=4,
        scoring='r2',
        n_jobs=-1
    )

    return scores.mean()

et_study = optuna.create_study(direction="maximize")

et_study.optimize(et_objective, n_trials=50)

print("========== ET BEST ==========")
print(et_study.best_params)
print(f"Best R²: {et_study.best_value:.4f}")

[I 2026-05-05 22:54:57,241] A new study created in memory with name: no-name-d3e28255-15d8-444e-b2f0-4cee500920cc
[I 2026-05-05 22:54:59,553] Trial 0 finished with value: 0.4231453310231973 and parameters: {'n_estimators': 273, 'max_depth': 22, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 0 with value: 0.4231453310231973.
[I 2026-05-05 22:55:00,997] Trial 1 finished with value: 0.428125175757086 and parameters: {'n_estimators': 154, 'max_depth': 35, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 1 with value: 0.428125175757086.
[I 2026-05-05 22:55:02,534] Trial 2 finished with value: 0.5765694907284223 and parameters: {'n_estimators': 99, 'max_depth': 21, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 2 with value: 0.5765694907284223.
[I 2026-05-05 22:55:02,958] Trial 3 finished with value: 0.4735207474424815 and parameters: {'n_estimators': 107, 'max_depth': 35, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 2 with value: 0

========== ET BEST ==========
{'n_estimators': 281, 'max_depth': 25, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Best R²: 0.5960


## 8. USE Baseline

In [17]:
et = ExtraTreesRegressor(
    n_estimators=281,
    max_depth=25,
    min_samples_leaf=1,
    max_features='sqrt',
    criterion='squared_error',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

et.fit(X_train, y_train)
print("Model trained.")

Model trained.


## 9. Evaluate Model

In [18]:
y_pred = et.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("=" * 50)
print("BASELINE RESULTS (single seed = 42)")
print("=" * 50)
print(f"R²    = {r2:.4f}    (paper: 0.25)")
print(f"RMSE  = {rmse:.4f}    (paper: 7.91)")
print(f"MAE   = {mae:.4f}")
print(f"Accuracy (100 - MAE) = {100 - mae:.2f}%   (paper: 96%)")
print(f"\n⚠️  Single-seed result อาจแกว่ง — ดู multi-seed check ด้านล่าง")

BASELINE RESULTS (single seed = 42)
R²    = 0.6615    (paper: 0.25)
RMSE  = 12.2776    (paper: 7.91)
MAE   = 5.7253
Accuracy (100 - MAE) = 94.27%   (paper: 96%)

⚠️  Single-seed result อาจแกว่ง — ดู multi-seed check ด้านล่าง


## 10. K-Fold Cross Validation (k=4)


In [19]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

cv = KFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)

# ✅ Wrap ใน Pipeline เพื่อให้ imputer fit ใหม่ทุก fold
et_cv = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', ExtraTreesRegressor(
        n_estimators=281,
        max_depth=25,
        min_samples_leaf=1,
        max_features='sqrt',
        criterion='squared_error',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

cv_results = cross_validate(
    et_cv,
    X,
    y,
    cv=cv,
    scoring={
        'r2': 'r2',
        'neg_mse': 'neg_mean_squared_error',
        'neg_mae': 'neg_mean_absolute_error'
    },
    n_jobs=-1,
    return_train_score=False
)


kfold_df = pd.DataFrame({
    'fold': range(1, 5),
    'R²': cv_results['test_r2'],
    'RMSE': np.sqrt(-cv_results['test_neg_mse']),
    'MAE': -cv_results['test_neg_mae']
})
kfold_df['Accuracy'] = 100 - kfold_df['MAE']

print("K-Fold Cross Validation (k=4):")
print(kfold_df.to_string(index=False))
print()
print(f"Mean R²:       {kfold_df['R²'].mean():.4f} ± {kfold_df['R²'].std():.4f}")
print(f"Mean RMSE:     {kfold_df['RMSE'].mean():.4f} ± {kfold_df['RMSE'].std():.4f}")
print(f"Mean MAE:      {kfold_df['MAE'].mean():.4f} ± {kfold_df['MAE'].std():.4f}")
print(f"Mean Accuracy: {kfold_df['Accuracy'].mean():.2f}% ± {kfold_df['Accuracy'].std():.2f}%")

K-Fold Cross Validation (k=4):
 fold       R²      RMSE      MAE  Accuracy
    1 0.641294 12.582752 5.939256 94.060744
    2 0.634526 13.167786 6.136044 93.863956
    3 0.668241 12.445471 5.988883 94.011117
    4 0.613905 13.511379 6.398724 93.601276

Mean R²:       0.6395 ± 0.0224
Mean RMSE:     12.9268 ± 0.5000
Mean MAE:      6.1157 ± 0.2063
Mean Accuracy: 93.88% ± 0.21%
